# Run the EEG pipeline

Run the cells in order. The installation cell modifies the selected kernel's environment, or your user site-packages if the kernel uses the shared system Python. Start Jupyter in a Valeria GPU session after loading the Python and CUDA modules. Loading modules in a notebook cell does not configure an already running kernel.

The repository and EEG data must already be present on the machine. The pinned requirements were recorded on Valeria and are not guaranteed to install on Colab or other platforms. Installation downloads and runs the Alliance wheel-conversion tool and requires its utilities, the Valeria wheelhouse, GitHub and PyPI. The pipeline can take many hours; keep the session active.

In [ ]:
from pathlib import Path
import subprocess
import sys

# Change this if the notebook was not started from the repository root.
repo = Path.cwd().resolve()
pipeline_args = ["--workers", "1"]  # Add "--rapide" only for a reduced test.

for name in ("requirement.txt", "00_pipeline.py", "00_notebook_install.py"):
    if not (repo / name).is_file():
        raise FileNotFoundError(f"{name} not found in {repo}; set repo to the project root.")

print(f"Repository: {repo}")
print(f"Python kernel: {sys.executable}")

def run_command(command):
    print("Running:", subprocess.list2cmdline([str(arg) for arg in command]), flush=True)
    with subprocess.Popen(command, cwd=repo, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding="utf-8", errors="replace", bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

## Install and check requirements

This converts the pinned CuPy wheel with the Alliance tool, then installs all pinned requirements and checks CuPy's import. It can fail if conversion tools, a wheel, or a dependency are unavailable. Restart the kernel after installation, then run the first cell again before starting the pipeline.

In [ ]:
run_command([sys.executable, "-u", str(repo / "00_notebook_install.py"), str(repo / "requirement.txt")])

## Run the pipeline

The launcher detects available GPU backends and reports CPU fallbacks. Results and logs are saved under `sorties`. A failed command stops the cell and reports its exit code.

In [ ]:
run_command([sys.executable, "-u", str(repo / "00_pipeline.py"), *pipeline_args])